# MidMamba Local Baseline Smoke

Run this notebook from your local Jupyter environment to verify the repo, find a local Databento `.dbn.zst` file, run immediate/TWAP baselines, and smoke-test the execution environment.

The unit tests do not require DBN files. The baseline cells require at least one `.dbn.zst` file under `data/` or a full path assigned to `DBN_FILE`.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "midmamba").exists():
            return candidate
    raise RuntimeError("Could not find the midmamba repo root. Start Jupyter from the repo or edit REPO_ROOT manually.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("repo:", REPO_ROOT)
print("python:", sys.executable)

repo: /Users/ak/Documents/genaiexperiments/midmamba
python: /opt/anaconda3/bin/python


In [2]:
def run(cmd, *, check=True):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    env = {**os.environ, "PYTHONPATH": src_path + os.pathsep + os.environ.get("PYTHONPATH", "")}
    proc = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with return code {proc.returncode}")
    return proc.returncode

## 1. Run Unit Tests

In [3]:
run([sys.executable, "-m", "pytest", "tests", "-q"])

$ /opt/anaconda3/bin/python -m pytest tests -q
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: /Users/ak/Documents/genaiexperiments/midmamba
configfile: pyproject.toml
plugins: anyio-4.10.0
collected 27 items

tests/test_baselines.py ....                                             [ 14%]
tests/test_execution_env.py .........                                    [ 48%]
tests/test_lob_mamba.py .....                                            [ 66%]
tests/test_mbp10_features.py ....                                        [ 81%]
tests/test_window_loader.py .....                                        [100%]

============================== 27 passed in 1.94s ==============================



0

## 2. Find a Local DBN File

If nothing is found, set `DBN_FILE = Path("/full/path/to/file.dbn.zst")` in the next cell.

In [4]:
dbn_files = sorted(REPO_ROOT.glob("data/**/*.dbn.zst"))
print(f"found {len(dbn_files)} DBN files")
for path in dbn_files[:10]:
    print(path)

DBN_FILE = dbn_files[0] if dbn_files else None
DBN_FILE

found 42 DBN files
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250304.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250305.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250306.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250307.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250310.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250311.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250312.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250313.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250314.mbp-10.dbn.zst


PosixPath('/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst')

## 3. Baseline Parameters

In [5]:
SAMPLE_ROWS = 10_000
WINDOW_STEPS = 1_000
PARENT_QUANTITY = 10_000
TWAP_SLICES = 20
SIDE = "buy"
OUTPUT_JSON = REPO_ROOT / "results" / "baseline_smoke_local.json"

print("DBN_FILE:", DBN_FILE)
print("output:", OUTPUT_JSON)

DBN_FILE: /Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
output: /Users/ak/Documents/genaiexperiments/midmamba/results/baseline_smoke_local.json


## 4. Run Immediate and TWAP Baselines

In [6]:
if DBN_FILE is None:
    raise FileNotFoundError("No DBN file found. Put a .dbn.zst under data/ or set DBN_FILE to a full path.")

run([
    sys.executable,
    "scripts/run_baseline_smoke.py",
    "--dbn-file", DBN_FILE,
    "--sample-rows", SAMPLE_ROWS,
    "--window-steps", WINDOW_STEPS,
    "--parent-quantity", PARENT_QUANTITY,
    "--twap-slices", TWAP_SLICES,
    "--side", SIDE,
    "--output-json", OUTPUT_JSON,
])

$ /opt/anaconda3/bin/python scripts/run_baseline_smoke.py --dbn-file /Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst --sample-rows 10000 --window-steps 1000 --parent-quantity 10000 --twap-slices 20 --side buy --output-json /Users/ak/Documents/genaiexperiments/midmamba/results/baseline_smoke_local.json
[baseline] loading /Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
[baseline] sample_rows=10000 window_steps=1000 start=0
Traceback (most recent call last):
  File "/Users/ak/Documents/genaiexperiments/midmamba/scripts/run_baseline_smoke.py", line 78, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/Users/ak/Documents/genaiexperiments/midmamba/scripts/run_baseline_smoke.py", line 46, in main
    loader = MBP10WindowLoader.from_dbn_file(dbn_file, sample_rows=args.sample_rows, seed=args.seed)
  File "/Users/ak/Documents/genaiexperiments/midmamba/src/midmamba/data/window

RuntimeError: command failed with return code 1

In [ ]:
report = json.loads(OUTPUT_JSON.read_text())
report["baselines"]

## 5. Smoke-Test the PPO-Facing Environment

In [ ]:
import numpy as np

from midmamba.data import MBP10WindowLoader
from midmamba.env import MidMambaExecutionEnv

loader = MBP10WindowLoader.from_dbn_file(DBN_FILE, sample_rows=min(SAMPLE_ROWS, 5_000), seed=1)
env = MidMambaExecutionEnv(loader, execution_steps=60, initial_inventory=float(PARENT_QUANTITY), side=SIDE)

obs, info = env.reset()
print("obs_shape:", obs.shape)
print("reset_info:", info)

# Try one fully aggressive market action.
next_obs, reward, terminated, truncated, step_info = env.step(np.array([1.0, 1.0], dtype=np.float32))
print("next_obs_shape:", next_obs.shape)
print("reward:", reward)
print("terminated:", terminated, "truncated:", truncated)
step_info

## Next

If this notebook passes, the next project step is adding the PPO rollout buffer and `train_ppo_smoke.py`.